## Imports

In [107]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor

import kaggle_evaluation.default_inference_server

## Project Directory Structure

In [108]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

## Configurations

In [109]:
# ============ PATHS ============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                                            # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                                            # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                                   # Multiplier of the OLS market forward excess returns predictions to signal 

# ============ MODEL CONFIGS ============
CV = 5                                                            # Number of cross validation folds in the model fitting
L1_RATIO: np.ndarray = np.linspace(0.01,0.99,10)                   # ElasticNet mixing parameters
ALPHAS: np.ndarray = np.logspace(-4, -1, 100)                       # Constant that multiplies the penalty terms
MAX_ITER: int = 1000000                                            # The maximum number of iterations
RANDOM_STATE: int = 17                                             # Random state

## Dataclasses Helpers

In [110]:
@dataclass
class DatasetOutput:
    X : pl.DataFrame 
    y: pl.Series
    scaler: StandardScaler

@dataclass 
class ElasticNetParameters:
    l1_ratio : np.ndarray 
    cv: int
    alphas: np.ndarray 
    max_iter: int
    random_state: int
    
    def __post_init__(self): 
        if self.l1_ratio.any() < 0 or self.l1_ratio.any() > 1: 
            raise ValueError("Wrong initializing value for ElasticNet l1_ratio")
        
@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL

## Set the Parameters

In [111]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

enet_params = ElasticNetParameters(
    l1_ratio = L1_RATIO, 
    cv = CV, 
    alphas = ALPHAS, 
    max_iter = MAX_ITER,
    random_state = RANDOM_STATE
)

## Dataset Loading/Creating Helper Functions

In [112]:
def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """

    vars_to_keep: List[str] = [
        "S2", "E19", "M17", "E11", "E18", "E12", "M18", "I2", 
        "E6", "M4", "I9", "E9", "E16", "I5", "M10", "P10",
        "E17", "M8", "E15", "M5", "M12", "P8", "S5", "U1", "U2"
    ]

    vars_to_use: List[str] = [ 
        'D1', 'D2', 'E10', 'E19', 'I2', 'M12', 'M17', 'M2', 'M3', 
        'M4', 'P1', 'P10', 'P11', 'P12', 'P3', 'P6', 'P7', 'P8', 
        'S2', 'S5', 'V13', 'V7', 'U1'
    ]

    vars_to_try: List[str] = [
        'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 
        'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 
        'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 
        'E6', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 
        'I6', 'I7', 'I8', 'I9', 'M10', 'M11', 'M12', 
        'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 
        'M5', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 
        'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 
        'P8', 'P9', 'S1', 'S10', 'S11', 'S2', 'S4', 
        'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V11', 'V12', 
        'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8',
        'U1', 'U2'
    ]
    
    all_vars: List[str] = [
        'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 
        'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 
        'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 
        'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 
        'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13', 
        'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 
        'M5', 'M6', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 
        'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 
        'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4', 
        'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 
        'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9',
        'U1', 'U2'
    ]

    # lags = [1, 2, 5, 10]
    # windows = [5, 20]

    # return (
    #     df.with_columns(
    #     (pl.col("I2") - pl.col("I1")).alias("U1"),
    #     (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
    #     )
    #     .with_columns([
    #         pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5)).forward_fill()
    #         for col in vars_to_try
    #     ])
    #     .with_columns([
    #         pl.col("target").shift(lag).alias(f"target_lag{lag}")
    #         for lag in lags
    #     ])
    #     .with_columns([
    #         pl.col("target").rolling_mean(window_size=w).shift(1).alias(f"ret_mean_{w}")
    #         for w in windows
    #     ] + [
    #         pl.col("target").rolling_std(window_size=w).shift(1).alias(f"vol_{w}")
    #         for w in windows
    #     ])
    #     .select(["date_id", "target"] + vars_to_try +
    #             [f"target_lag{lag}" for lag in lags] +
    #             [f"ret_mean_{w}" for w in windows] +
    #             [f"vol_{w}" for w in windows])
    #     .drop_nulls()
    # )
    
    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5)).forward_fill()
            for col in vars_to_try
        ])
        .select(["date_id", "target"] + vars_to_try)
        .drop_nulls()
    )
 
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(df: pl.DataFrame, features: list[str]) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        df (pl.DataFrame): The processed  DataFrame.
        features (list[str]): List of features to used in model. 

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X = df.drop(['date_id','target']) 
    y = df.get_column('target')
    
    scaler = StandardScaler() 
    
    X_scaled_np = scaler.fit_transform(X)
    X = pl.from_numpy(X_scaled_np, schema=features)
    
    
    return DatasetOutput(
        X = X,
        y = y, 
        scaler = scaler
    )

## Converting Return Prediction to Signal

Here is an example of a potential function used to convert a prediction based on the market forward excess return to a daily signal position. 

In [113]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

## Looking at the Data

In [114]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 
print(train.tail(3)) 
print(test.head(3))

shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬──────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target   │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---      │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64      │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪══════════╡
│ 8977    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.708599 ┆ 0.004187        ┆ 0.000162       ┆ 0.003713 │
│ 8978    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.725858 ┆ 0.002279        ┆ 0.000162       ┆ 0.001805 │
│ 8979    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.720092 ┆ 0.003541        ┆ 0.000161       ┆ 0.003068 │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴──────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scor

In [115]:
# train_df = pl.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
# null_counts = train_df.select(pl.all().null_count())
# null_count_dict = null_counts.row(0)  
# columns_with_many_nulls = [
#     col for col, count in zip(null_counts.columns, null_count_dict) if count > 1006
# ]

# print(columns_with_many_nulls)

## Generating the Train and Test

In [116]:
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 
train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES: list[str] = [col for col in test.columns if col not in ['date_id', 'target']]

dataset: DatasetOutput = split_dataset(df=df, features=FEATURES) 

X: pl.DataFrame = dataset.X
y: pl.Series = dataset.y
scaler: StandardScaler = dataset.scaler 

## Fitting the Model 

In [119]:
def evaluate_preds(y_true, preds):
    if hasattr(y_true, 'to_numpy'):
        y_true = y_true.to_numpy().flatten() 
    
    r2 = r2_score(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    corr = np.corrcoef(y_true, preds)[0,1]
    direction_acc = (np.sign(preds) == np.sign(y_true)).mean()
    strategy_returns = preds * y_true
    ir = np.mean(strategy_returns) / np.std(strategy_returns) if np.std(strategy_returns) > 0 else 0
    return dict(R2=r2, RMSE=rmse, Corr=corr, DirectionAcc=direction_acc, IR=ir)

In [120]:
X_np = X.to_numpy()
y_np = y.to_numpy().flatten()

tscv = TimeSeriesSplit(n_splits=enet_params.cv)  

all_metrics = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_np, y_np), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model_cv = ElasticNetCV(
        **asdict(enet_params)  
    )
    model_cv.fit(X_train, y_train)

    model = ElasticNet(
        alpha=model_cv.alpha_,
        l1_ratio=model_cv.l1_ratio_,
        random_state=enet_params.random_state
    )
    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    test_preds  = model.predict(X_test)

    metrics_train = evaluate_preds(y_train, train_preds)
    metrics_test  = evaluate_preds(y_test, test_preds)

    print(f"\nFold {fold}")
    print(f"  Best alpha: {model_cv.alpha_:.6f}, l1_ratio: {model_cv.l1_ratio_:.3f}")
    print("  Train metrics:", metrics_train)
    print("  Test metrics: ", metrics_test)

    all_metrics.append(metrics_test)

df_metrics = pl.DataFrame(all_metrics)
print("\nAverage test metrics across folds:")
print(df_metrics.mean())


Fold 1
  Best alpha: 0.000933, l1_ratio: 0.337
  Train metrics: {'R2': 0.007841740032855538, 'RMSE': 0.0077613619309085035, 'Corr': 0.11997501578719272, 'DirectionAcc': 0.5388655462184874, 'IR': 0.1060068262910054}
  Test metrics:  {'R2': 0.0007123139172577897, 'RMSE': 0.014505360002960892, 'Corr': 0.07234724018182473, 'DirectionAcc': 0.526813880126183, 'IR': 0.02770169401926184}

Fold 2
  Best alpha: 0.053367, l1_ratio: 0.010
  Train metrics: {'R2': 0.01169704772137059, 'RMSE': 0.011578230929068964, 'Corr': 0.1408130180702084, 'DirectionAcc': 0.512348922753547, 'IR': 0.08663974071236842}
  Test metrics:  {'R2': 0.0025975776925813987, 'RMSE': 0.009671215842957055, 'Corr': 0.07879230942830827, 'DirectionAcc': 0.49737118822292326, 'IR': 0.04467306850765045}

Fold 3
  Best alpha: 0.000433, l1_ratio: 0.772
  Train metrics: {'R2': 0.014755972507952464, 'RMSE': 0.010952494879219388, 'Corr': 0.14202020486678477, 'DirectionAcc': 0.524176594253679, 'IR': 0.06977207808972313}
  Test metrics:  {

In [ ]:
# model_cv: ElasticNetCV = ElasticNetCV(
#     **asdict(enet_params)
# )
# model_cv.fit(X_train, y_train) 
        
# # Fit the final model using the best alpha found by cross-validation
# model: ElasticNet = ElasticNet(
#                                alpha=model_cv.alpha_, 
#                                l1_ratio=model_cv.l1_ratio_, 
#                                random_state=enet_params.random_state
#                               ) 
# model.fit(X_train, y_train)

In [ ]:
# k_best = SelectKBest(score_func=f_regression, k=20)

# X_train_filtered = k_best.fit_transform(X_train, y_train)

# selected_features = k_best.get_feature_names_out()

# print(f"Original number of features: {X_train.shape[1]}")
# print(f"Filtered number of features: {X_train_filtered.shape[1]}")
# print("\nSelected features:")
# print(selected_features)

In [ ]:
# rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
# rf.fit(X_train, y_train)

In [ ]:
# importances = pl.DataFrame({'features' : X_train.columns, 'importance' : rf.feature_importances_})
# top_features = importances.sort(by='importance', descending=True).head(25)
# with pl.Config(tbl_rows=50):
#     print(top_features)

## Prediction Function via Kaggle Server

In [121]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df: pl.DataFrame = create_example_dataset(test)
    X_test: pl.DataFrame = df.select(FEATURES)
    X_test_scaled_np: np.ndarray = scaler.transform(X_test)
    X_test: pl.DataFrame = pl.from_numpy(X_test_scaled_np, schema=FEATURES)
    raw_pred: float = model.predict(X_test)[0]
    return convert_ret_to_signal(raw_pred, ret_signal_params)

## Launch Server

In [122]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))